# Phases 3 et 4 — fine-tuning, puis cascade contre bout-en-bout

1. **Whisper multi-tâche** sur Jeli-ASR (`--task both`) : un seul modèle,
   un seul encodeur, deux sorties. `<|transcribe|>` donne le bambara (la
   cascade), `<|translate|>` le français (le bout-en-bout).
2. **NLLB**, un modèle par sens, sur Bayelemabaga.
3. **Évaluation** avec exactement le protocole de la phase 1 (même
   partition, même graine, mêmes 300 énoncés), puis un tableau commun avec
   les références zero-shot.

**Durée.** Le fine-tuning de Whisper prend plusieurs heures sur un T4 (de
l'ordre d'une à deux heures par époque, pour trois époques), NLLB moins
d'une heure par sens. Une session Colab gratuite sera probablement coupée
en route : les checkpoints sont écrits sur Drive, et `--resume` repart du
dernier. Après une coupure, relancer les cellules 1 à 3, puis la cellule
d'entraînement interrompue.

*Exécution → Modifier le type d'exécution → GPU (T4).*

## 1. Installation

In [ ]:
# Branche de travail ; remplacer par "main" une fois la branche fusionnée.
BRANCH = "claude/chat-ia-bambara-dho8oa"

!git clone -q -b $BRANCH https://github.com/LOSTHADY/IA_Project- projet
%cd projet
!pip install -q transformers datasets accelerate sentencepiece jiwer sacrebleu soundfile librosa

## 2. Dossiers sur Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

CKPT = "/content/drive/MyDrive/bambara/modeles"            # checkpoints et modèles finaux
OUT = "/content/drive/MyDrive/bambara/resultats-phase3"    # rapports d'évaluation
PHASE1 = "/content/drive/MyDrive/bambara-phase1"           # rapports du notebook de la phase 1
!mkdir -p "$CKPT" "$OUT"

## 3. Accès aux jeux et sous-ensemble Jeli-ASR

Même réglage qu'en phase 1. Si Jeli-ASR demande un sous-ensemble, reprendre
celui choisi alors : les chiffres ne se comparent que sur le même jeu.

In [ ]:
JELI_CONFIG = ""  # ex. "bam-asr-oza", comme en phase 1

JELI = "--dataset RobotsMali/jeli-asr" + (f" --dataset-config {JELI_CONFIG}" if JELI_CONFIG else "")
JELI_EVAL = "--dataset RobotsMali/jeli-asr" + (f" --config {JELI_CONFIG}" if JELI_CONFIG else "")

# Si un jeu est « gated » : décommenter.
# from huggingface_hub import notebook_login; notebook_login()

## 4. Essai rapide (5 minutes)

200 exemples, une époque. Sert à vérifier que tout s'enchaîne (données,
colonnes, mémoire GPU) avant de lancer des heures de calcul. Les scores
de cet essai ne veulent rien dire.

In [ ]:
!python scripts/finetune_whisper.py $JELI --model openai/whisper-small --task both \
    --max-samples 200 --epochs 1 --save-steps 50 --eval-samples 20 --output /content/essai

## 5. Fine-tuning de Whisper (long)

Le meilleur checkpoint est choisi au WER sur une validation prise dans
`train`. La partition de test reste intacte pour l'évaluation (§7).
Relancer cette cellule après une coupure : elle reprend au dernier
checkpoint.

In [ ]:
!python scripts/finetune_whisper.py $JELI --model openai/whisper-small --task both \
    --output "$CKPT/whisper-small-bm" --resume

## 6. Fine-tuning de NLLB, un modèle par sens

In [ ]:
!python scripts/finetune_nllb.py --direction bm2fr --output "$CKPT/nllb-bm2fr" --resume

In [ ]:
!python scripts/finetune_nllb.py --direction fr2bm --output "$CKPT/nllb-fr2bm" --resume

## 7. Évaluation — cascade contre bout-en-bout

Même partition, même graine et même nombre d'énoncés qu'en phase 1. Les
deux architectures utilisent **le même Whisper** : seule la tâche change.

In [ ]:
!python -m eval.baselines --out "$OUT" --label "cascade affinée" \
    asr $JELI_EVAL --model "$CKPT/whisper-small-bm" \
    --with-mt --mt-model "$CKPT/nllb-bm2fr" --limit 300

In [ ]:
!python -m eval.baselines --out "$OUT" --label "bout-en-bout affiné" \
    asr $JELI_EVAL --model "$CKPT/whisper-small-bm" --task translate --limit 300

Pour séparer l'apport de chaque fine-tuning : la cascade avec le Whisper
affiné mais le NLLB d'origine. Facultatif.

In [ ]:
!python -m eval.baselines --out "$OUT" --label "cascade, NLLB d'origine" \
    asr $JELI_EVAL --model "$CKPT/whisper-small-bm" --with-mt --limit 300

### Traduction seule, dans les deux sens

In [ ]:
!python -m eval.baselines --out "$OUT" --label "NLLB affiné / Bayelemabaga" \
    mt --dataset RobotsMaliAI/bayelemabaga --limit 500 \
    --bm2fr-model "$CKPT/nllb-bm2fr" --fr2bm-model "$CKPT/nllb-fr2bm"

In [ ]:
!python -m eval.baselines --out "$OUT" --label "NLLB affiné / Jeli-ASR" \
    mt $JELI_EVAL --limit 500 \
    --bm2fr-model "$CKPT/nllb-bm2fr" --fr2bm-model "$CKPT/nllb-fr2bm"

## 8. Tableau commun : zero-shot (phase 1) et modèles affinés

In [ ]:
!python -m eval.compare "$PHASE1"/*.json "$OUT"/*.json

### L'écart entre les architectures est-il réel ?

Intervalles de confiance à 95 % et test apparié par bootstrap, sur les mêmes
énoncés. Si l'IC de l'écart contient zéro, la différence n'est pas établie :
c'est ce qu'il faut écrire dans le mémoire, quelle que soit la taille
apparente de l'écart.

In [ ]:
!python -m eval.significance "$OUT"/*.json --noms "cascade affinée" "bout-en-bout affiné"

### Comment les modèles affinés se trompent-ils ?

Même analyse d'erreurs que pour la phase 1 (`docs/RESULTATS.md`). À
vérifier en premier : le mot swahili « kwa » et les boucles de répétition,
qui expliquaient l'essentiel des insertions de Whisper zero-shot, doivent
avoir disparu. Le mot « kwa » trahirait un token de langue mal réaffecté
au bambara.

In [ ]:
!python -m eval.analysis "$OUT"/*.json | tee "$OUT/analyse.md"

## Lire les résultats

- **La comparaison centrale** est la ligne *chrF++ (depuis ASR)* : le
  français obtenu par la cascade (ASR puis NLLB) contre celui que Whisper
  produit directement, sur les mêmes énoncés et la même référence.
- **Propagation d'erreurs** (cascade) : ce que les erreurs d'ASR coûtent à
  la traduction. C'est le coût que le bout-en-bout cherche à supprimer.
  S'il reste faible, le maillon supplémentaire de la cascade est peu
  pénalisant, et elle garde ses avantages : texte bambara affichable et
  gabarits utilisables.
- **Latence** : mesurée sur GPU, donc indicative seulement. Le budget de
  déploiement se mesure sur CPU (phase 5).
- **Limites à écrire dans le mémoire** : Jeli-ASR est à la fois la source
  d'entraînement et d'évaluation. Si des locuteurs de test apparaissent
  aussi dans `train` (à vérifier dans la documentation du jeu), les scores
  sont optimistes. C'est pourquoi les chiffres définitifs se mesurent sur le
  jeu de test maison (phase 2), avec `eval.run_eval`.

## 9. Préparer le déploiement sur CPU (phase 5)

Convertit les trois modèles en int8 (CTranslate2) sur Drive, avec le
fichier `config.json` qui les décrit. Télécharger ensuite le dossier
`modeles-cpu` sur la machine de déploiement : voir `docs/DEPLOIEMENT.md`
pour le LLM en GGUF, la perte de qualité à mesurer et la latence.

In [ ]:
!pip install -q ctranslate2
!python scripts/export_cpu.py --out "$CKPT/../modeles-cpu" \
    --whisper "$CKPT/whisper-small-bm" \
    --nllb-bm2fr "$CKPT/nllb-bm2fr" --nllb-fr2bm "$CKPT/nllb-fr2bm"

## Facultatif — publier les modèles (en privé)

Pour les réutiliser ailleurs (démo, phase 5) sans passer par Drive.

In [ ]:
# from huggingface_hub import HfApi
# api = HfApi()
# for name in ("whisper-small-bm", "nllb-bm2fr", "nllb-fr2bm"):
#     repo = api.create_repo(f"VOTRE_COMPTE/{name}", private=True, exist_ok=True).repo_id
#     api.upload_folder(folder_path=f"{CKPT}/{name}", repo_id=repo,
#                       ignore_patterns=["checkpoint-*"])